# Cross-Station Comparison
## Overview
This notebook compares flood-frequency results for:

- USGS 06191500
- USGS 06192500
- USGS 06214500

It combines station-level results, identifies the best distributions, compares design floods, and saves summary tables and figures.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:.3f}".format

In [ ]:
# Define project folders and stations
OUTPUT_DIR = Path("outputs")
CROSS_STATION_DIR = OUTPUT_DIR / "cross_station"
FIGURE_DIR = CROSS_STATION_DIR / "figures"

CROSS_STATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

stations = {
    "06191500": OUTPUT_DIR / "usgs_06191500",
    "06192500": OUTPUT_DIR / "usgs_06192500",
    "06214500": OUTPUT_DIR / "usgs_06214500"
}

print(stations)

## 1. Read and combine station results

In [ ]:
def combine_station_files(filename):
    tables = []

    for station, folder in stations.items():
        file_path = folder / filename

        if not file_path.exists():
            raise FileNotFoundError(f"Missing file: {file_path}")

        table = pd.read_csv(file_path)
        table.insert(0, "Station", station)
        tables.append(table)

    return pd.concat(tables, ignore_index=True)


combined_gof = combine_station_files("goodness_of_fit_results.csv")
combined_rankings = combine_station_files("distribution_ranking.csv")
combined_design_floods = combine_station_files("design_flood_estimates.csv")

print("Files combined successfully.")

In [ ]:
combined_gof.to_csv(
    CROSS_STATION_DIR / "combined_goodness_of_fit_results.csv",
    index=False,
)

combined_rankings.to_csv(
    CROSS_STATION_DIR / "combined_distribution_rankings.csv",
    index=False,
)

combined_design_floods.to_csv(
    CROSS_STATION_DIR / "combined_design_flood_estimates.csv",
    index=False,
)

combined_rankings.head()

## 2. Identify the best distribution for each station

In [ ]:
summary_rows = []

for station in stations:
    rank_data = combined_rankings[combined_rankings["Station"] == station]
    gof_data = combined_gof[combined_gof["Station"] == station]

    minimum_rank = rank_data["Average Rank"].min()
    best_distributions = rank_data.loc[
        rank_data["Average Rank"].eq(minimum_rank), "Distribution"
    ].tolist()

    best_ks = gof_data.loc[gof_data["KS Statistic"].idxmin(), "Distribution"]
    lowest_rmse = gof_data.loc[gof_data["RMSE"].idxmin(), "Distribution"]

    summary_rows.append({
        "Station": station,
        "Best Overall Distribution(s)": " / ".join(best_distributions),
        "Minimum Average Rank": minimum_rank,
        "Best KS Distribution": best_ks,
        "Lowest RMSE Distribution": lowest_rmse,
        "All KS p-values > 0.05": (gof_data["KS p-value"] > 0.05).all(),
        "All Chi-Square p-values > 0.05": (
            gof_data["Chi-Square p-value"] > 0.05
        ).all(),
    })

station_summary = pd.DataFrame(summary_rows)
station_summary.to_csv(
    CROSS_STATION_DIR / "multi_station_summary.csv",
    index=False,
)

station_summary

## 3. Compare average distribution rankings

In [ ]:
mean_ranking = (
    combined_rankings
    .groupby("Distribution", as_index=False)["Average Rank"]
    .mean()
    .rename(columns={"Average Rank": "Mean Average Rank"})
    .sort_values("Mean Average Rank")
)

mean_ranking

In [ ]:
ax = mean_ranking.plot(
    x="Distribution",
    y="Mean Average Rank",
    kind="bar",
    legend=False,
    figsize=(9, 5),
)

ax.set_title("Average Distribution Ranking Across Stations")
ax.set_xlabel("Distribution")
ax.set_ylabel("Mean Average Rank (Lower is Better)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cross_station_average_rank.png", dpi=300)
plt.show()

## 4. Compare 100-year design-flood estimates

In [ ]:
distribution_columns = [
    "Normal",
    "Log-Normal",
    "Gumbel",
    "Log-Pearson Type III",
]

q100 = combined_design_floods.loc[
    combined_design_floods["Return Period (Years)"].eq(100),
    ["Station", *distribution_columns],
].set_index("Station")

q100

In [ ]:
ax = q100.plot(kind="bar", figsize=(10, 6))

ax.set_title("100-Year Design-Flood Estimates Across Stations")
ax.set_xlabel("USGS Station")
ax.set_ylabel("Peak Flow Discharge (Dataset Units)")
plt.xticks(rotation=0)
plt.legend(title="Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "q100_distribution_comparison.png", dpi=300)
plt.show()

## 5. Compare Log-Normal design floods across stations

In [ ]:
for station in stations:
    station_data = combined_design_floods[
        combined_design_floods["Station"] == station
    ]

    plt.plot(
        station_data["Return Period (Years)"],
        station_data["Log-Normal"],
        marker="o",
        label=station,
    )

plt.title("Log-Normal Design-Flood Comparison")
plt.xlabel("Return Period (Years)")
plt.ylabel("Peak Flow Discharge (Dataset Units)")
plt.xscale("log")
plt.grid(alpha=0.3)
plt.legend(title="USGS Station")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "lognormal_cross_station_comparison.png", dpi=300)
plt.show()

## Main Findings

- Log-Normal is the most consistent distribution across the three stations.
- Log-Pearson Type III generally gives the lowest KS statistic.
- Log-Normal generally gives the lowest RMSE.
- Gumbel gives the highest estimates at longer return periods.
- Differences among distributions increase as the return period increases.
- All four distributions pass the selected goodness-of-fit tests at the 5% significance level.

In [ ]:
print("Cross-station analysis completed.")
print(f"Tables saved in: {CROSS_STATION_DIR}")
print(f"Figures saved in: {FIGURE_DIR}")